# WS-SP — Selective Prediction / Abstention (Step 2 runner)

Runs the **patched** `train_agentic.py` for **Fusion + R6 (ensemble+conformal)**, 3 held-out
countries × 5 seeds, M=10, so each test image gets `ens_std` + cavity saved to disk. Then it
runs `scripts/selective_prediction.py` to produce the risk-coverage table + figure.

**Attach the dataset** `mabdullahi454/tb-portals-cxr-pngs`, turn the **GPU on**, then Run All.
Downloads at the end: `ws_sp_preds.zip` (the per-image CSVs) and `ws_sp_results.zip`.

## 0 — Clone (branch `dl-sidework`, which has the WS-SP patch) + install

In [ ]:
import os, sys, subprocess
REPO_URL = 'https://github.com/mabdullahi7780/dl-project-codebase.git'
REPO_DIR = '/kaggle/working/dl-project-codebase'
BRANCH = 'dl-sidework'   # <-- has the ens_std/cavity per-image patch + selective_prediction.py
if os.path.isdir(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)
for p in (REPO_DIR, REPO_DIR + '/scripts'):
    if p not in sys.path: sys.path.insert(0, p)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers', 'matplotlib'], check=False)
print('ready')

## 1 — Paths + build the 5,010-image manifest

In [ ]:
import os, sys, pandas as pd
from pathlib import Path
WORK = '/kaggle/working'
REPO_DIR = '/kaggle/working/dl-project-codebase'
DATASET = '/kaggle/input/datasets/mabdullahi454/tb-portals-cxr-pngs'
KAGGLE_EXPORT = f'{DATASET}/kaggle_export'
PAPER_MANIFEST = f'{WORK}/tbportals_manifest_paper.csv'
FEATURES_CLS  = f'{WORK}/features_rad-dino_cls.npz'
PREDS_DIR = f'{WORK}/ws_sp/preds'
if REPO_DIR + '/scripts' not in sys.path: sys.path.insert(0, REPO_DIR + '/scripts')
from build_paper_manifest import subsample
raw = pd.read_csv(f'{KAGGLE_EXPORT}/manifest.csv', dtype={'image_id': str, 'patient_id': str, 'country': str})
raw['image_path'] = raw['image_path'].apply(lambda p: p if str(p).startswith('/') else f'{KAGGLE_EXPORT}/{p}')
paper_df = subsample(raw, seed=42)
paper_df['image_id'] = paper_df['image_path'].apply(lambda p: Path(str(p)).stem)
paper_df.to_csv(PAPER_MANIFEST, index=False)
print('manifest size:', len(paper_df))

## 2 — Cache RAD-DINO CLS features (one-time; idempotent). Fusion+R6 does not need the patch grid.

In [ ]:
from cache_features import main as cache_main
if os.path.isfile(FEATURES_CLS):
    print('cached ->', FEATURES_CLS)
else:
    cache_main(['--manifest', PAPER_MANIFEST, '--out', FEATURES_CLS,
                '--backbone', 'rad-dino', '--batch-size', '32'])

## 3 — Run Fusion + R6, 3 countries × 5 seeds, M=10
Per-image predictions (with `ens_std`, `cavity_*`) are written to `PREDS_DIR` automatically
as `preds_fusion_rung6_conformal_{country}_s{seed}.csv`.

In [ ]:
from src.training.train_agentic import main as agentic_main
os.makedirs(PREDS_DIR, exist_ok=True)
agentic_main(['--features', FEATURES_CLS, '--manifest', PAPER_MANIFEST,
              '--mode', 'fusion', '--rungs', '6', '--ensemble-m', '10',
              '--seeds', '0', '1', '2', '3', '4',
              '--held-outs', 'Romania', 'Moldova', 'Kazakhstan',
              '--out-dir', PREDS_DIR])
print('preds written:', sorted(p for p in os.listdir(PREDS_DIR) if p.startswith('preds_')))

## 4 — Selective-prediction analysis (risk-coverage table + figure)

In [ ]:
from selective_prediction import main as sp_main
sp_main(['--preds-dir', PREDS_DIR, '--mode', 'fusion', '--rung', 'rung6_conformal',
         '--out-dir', f'{WORK}/ws_sp'])

## 5 — Bundle for download

In [ ]:
import shutil
print('preds  ->', shutil.make_archive(f'{WORK}/ws_sp_preds', 'zip', PREDS_DIR))
print('results->', shutil.make_archive(f'{WORK}/ws_sp_results', 'zip', f'{WORK}/ws_sp'))